# ML-07 — Baseline Action Score and Top-10 Review

**Lane 2: Refresh / Content Opportunity Scoring**

This notebook freezes the transparent rule that the Week-5 model must beat. It uses the bundled 30,000-row pseudonymized starter slice and writes the ranked queue to `work/outputs/baseline_action_score.csv`.

## 1. Check two signals before writing the rule

**Signal 1 — staleness (behind FlyRank's refresh flags).** Hypothesis: pages not updated for more than 30 days are more likely to carry the decline proxy than recently updated pages. **Verdict: CONFIRMED.** The executed table below shows 60.8% versus 51.1%, with `n` printed.

**Signal 2 — search volume (behind FlyRank's quick-win logic).** Hypothesis: pages with at least 500 impressions have enough measured demand to deserve priority and are more likely to carry the decline proxy than low-volume pages. **Verdict: CONFIRMED.** The executed table shows 59.6% versus 47.5%, with `n` printed.

**The one rule, in plain words:** review a page when it is both stale and still visible in search. A page qualifies at `days_since_last_update > 30` and `impressions_90d >= 500`; eligible pages rank by staleness multiplied by log search demand. This is a prioritization rule, not a claim that refreshing causes recovery.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
DATA_PATH = DATA_REL if DATA_REL.exists() else Path("../..") / DATA_REL
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)

df["staleness_bucket"] = pd.Categorical(
    np.where(df["days_since_last_update"] > 30, "stale (>30d)", "recent (<=30d)"),
    categories=["recent (<=30d)", "stale (>30d)"], ordered=True,
)
staleness_table = (df.groupby("staleness_bucket", observed=False)
                   .agg(n=("is_declining_label", "size"),
                        declining_n=("is_declining_label", "sum"),
                        decline_rate=("is_declining_label", "mean"))
                   .reset_index())
staleness_table["decline_rate"] = staleness_table["decline_rate"].round(3)
print("SIGNAL 1 — STALENESS | VERDICT: CONFIRMED")
display(staleness_table)

df["volume_bucket"] = pd.Categorical(
    np.where(df["impressions_90d"] >= 500, "visible (>=500)", "low volume (<500)"),
    categories=["low volume (<500)", "visible (>=500)"], ordered=True,
)
volume_table = (df.groupby("volume_bucket", observed=False)
                .agg(n=("is_declining_label", "size"),
                     declining_n=("is_declining_label", "sum"),
                     decline_rate=("is_declining_label", "mean"))
                .reset_index())
volume_table["decline_rate"] = volume_table["decline_rate"].round(3)
print("SIGNAL 2 — SEARCH VOLUME | VERDICT: CONFIRMED")
display(volume_table)

SIGNAL 1 — STALENESS | VERDICT: CONFIRMED


,staleness_bucket,n,declining_n,decline_rate
0,recent (<=30d),20480,10473,0.511
1,stale (>30d),9520,5789,0.608


SIGNAL 2 — SEARCH VOLUME | VERDICT: CONFIRMED


,volume_bucket,n,declining_n,decline_rate
0,low volume (<500),13274,6301,0.475
1,visible (>=500),16726,9961,0.596


## 2. Build the ranked queue (writes the CSV)

The score has no learned weights. Every output row carries one `reason_code` and one `action_label`:

- Eligible: `stale_with_search_demand` → `review_for_refresh`
- Not eligible: `below_rule_threshold` → `monitor`

`trend_direction` is used only after ranking to evaluate Precision@K; it is never an input to eligibility, score, reason, or action.

In [2]:
RULE_INPUTS = ["days_since_last_update", "impressions_90d"]
FORBIDDEN_INPUTS = {
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}
assert FORBIDDEN_INPUTS.isdisjoint(RULE_INPUTS)

eligible = (df["days_since_last_update"] > 30) & (df["impressions_90d"] >= 500)
df["baseline_score"] = np.where(
    eligible,
    df["days_since_last_update"] * np.log1p(df["impressions_90d"]),
    0.0,
)
df["reason_code"] = np.where(eligible, "stale_with_search_demand", "below_rule_threshold")
df["action_label"] = np.where(eligible, "review_for_refresh", "monitor")

queue = df.sort_values(["baseline_score", "content_id"], ascending=[False, True], kind="stable").reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
base_rate = queue["is_declining_label"].mean()
precision_at_10 = queue.head(10)["is_declining_label"].mean()
precision_at_50 = queue.head(50)["is_declining_label"].mean()

OUTPUT_DIR = Path("work/outputs") if Path("work").exists() else Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUEUE_PATH = OUTPUT_DIR / "baseline_action_score.csv"
METRICS_PATH = OUTPUT_DIR / "baseline_metrics.json"
queue_columns = [
    "rank", "content_id", "client_id", "content_type",
    "days_since_last_update", "impressions_90d", "avg_position", "ctr",
    "baseline_score", "reason_code", "action_label",
]
queue[queue_columns].to_csv(QUEUE_PATH, index=False)
metrics = {
    "rows_ranked": int(len(queue)),
    "eligible_rows": int(eligible.sum()),
    "base_rate": round(float(base_rate), 4),
    "precision_at_10": round(float(precision_at_10), 4),
    "precision_at_50": round(float(precision_at_50), 4),
    "rule_inputs": RULE_INPUTS,
    "thresholds": {"days_since_last_update": ">30", "impressions_90d": ">=500"},
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")

print(f"Wrote {len(queue):,} ranked rows to {QUEUE_PATH}")
print(f"Eligible rows: {eligible.sum():,}")
print(f"Base decline rate: {base_rate:.3f}")
print(f"Precision@10: {precision_at_10:.3f}")
print(f"Precision@50: {precision_at_50:.3f}")
display(queue[queue_columns].head(10))

Wrote 30,000 ranked rows to work\outputs\baseline_action_score.csv
Eligible rows: 6,663
Base decline rate: 0.542
Precision@10: 1.000
Precision@50: 0.580


,rank,content_id,client_id,content_type,days_since_last_update,impressions_90d,avg_position,ctr,baseline_score,reason_code,action_label
0,1,content_cf56e2e2e282,client_7f2253d7e2,keyword article,194,61678,19.7,0.15,2139.761566,stale_with_search_demand,review_for_refresh
1,2,content_7368877ea310,client_7f2253d7e2,keyword article,194,59472,24.8,0.13,2132.695875,stale_with_search_demand,review_for_refresh
2,3,content_7f116ae1f6f5,client_9400f1b21c,keyword article,301,954,9.0,0.42,2065.375113,stale_with_search_demand,review_for_refresh
3,4,content_72496874f806,client_4ec9599fc2,keyword article,301,821,5.8,0.24,2020.233859,stale_with_search_demand,review_for_refresh
4,5,content_1bfaa38ff26c,client_7f2253d7e2,keyword article,194,25715,22.2,0.23,1970.044517,stale_with_search_demand,review_for_refresh
5,6,content_0a91db491d14,client_7f2253d7e2,keyword article,193,13299,10.5,0.49,1832.635228,stale_with_search_demand,review_for_refresh
6,7,content_5feee3994adb,client_7f2253d7e2,keyword article,194,7812,39.0,0.01,1738.927593,stale_with_search_demand,review_for_refresh
7,8,content_c2d929d83eaa,client_7f2253d7e2,keyword article,193,7558,17.9,0.20,1723.585378,stale_with_search_demand,review_for_refresh
8,9,content_b16bd7307b39,client_7f2253d7e2,keyword article,194,4590,31.0,0.00,1635.779510,stale_with_search_demand,review_for_refresh
9,10,content_fe16a55cd13d,client_7f2253d7e2,keyword article,194,4556,16.4,0.33,1634.337440,stale_with_search_demand,review_for_refresh


## 3. Top-10 review

Each line below states the action, why the row ranked, and a concrete condition that would make the recommendation wrong. The skeptical notes are decision risks, not extra scoring features.

In [3]:
top10 = queue.head(10).copy()
wrong_if = [
    "the traffic drop is seasonal and demand will return without an edit",
    "the page has been replaced or consolidated under another canonical page",
    "954 impressions are too little evidence and the apparent decline is noise",
    "a recent update was made but the update timestamp was not recorded",
    "the high-volume topic is intentionally retired despite its remaining demand",
    "the query market itself contracted, so refreshing cannot recover demand",
    "position 39 means ranking weakness, not stale content, is the main problem",
    "the page already satisfies intent and the decline comes from SERP changes",
    "zero CTR at position 31 is expected visibility loss, not proof of a refresh need",
    "the page serves a strategic purpose that search impressions do not measure",
]
top10["why_it_is_here"] = top10.apply(
    lambda r: f"{int(r.days_since_last_update)} days stale with {int(r.impressions_90d):,} impressions; reason={r.reason_code}",
    axis=1,
)
top10["what_would_make_it_wrong"] = wrong_if
for r in top10.itertuples():
    print(f"{r.rank}. Action: {r.action_label} | Why: {r.why_it_is_here} | Wrong if: {r.what_would_make_it_wrong}.")

1. Action: review_for_refresh | Why: 194 days stale with 61,678 impressions; reason=stale_with_search_demand | Wrong if: the traffic drop is seasonal and demand will return without an edit.
2. Action: review_for_refresh | Why: 194 days stale with 59,472 impressions; reason=stale_with_search_demand | Wrong if: the page has been replaced or consolidated under another canonical page.
3. Action: review_for_refresh | Why: 301 days stale with 954 impressions; reason=stale_with_search_demand | Wrong if: 954 impressions are too little evidence and the apparent decline is noise.
4. Action: review_for_refresh | Why: 301 days stale with 821 impressions; reason=stale_with_search_demand | Wrong if: a recent update was made but the update timestamp was not recorded.
5. Action: review_for_refresh | Why: 194 days stale with 25,715 impressions; reason=stale_with_search_demand | Wrong if: the high-volume topic is intentionally retired despite its remaining demand.
6. Action: review_for_refresh | Why: 19

## 4. Weak picks + leakage check

The top ten happen to match the decline proxy, but that does not make every proposed action sound. Ranks 7 and 9 are weak action candidates because their average positions are 39 and 31: low CTR or falling impressions there may reflect weak rankings rather than stale content. More broadly, Precision@50 is only slightly above the overall base rate, so this is an honestly beatable baseline—not a production policy.

The rule is now frozen for Week 5. A later model must use the same slice and evaluation metric, and must beat this receipt without adding label-derived or future-window inputs.

In [4]:
weak_picks = top10.loc[(top10["avg_position"] > 30) | (top10["ctr"] < 0.05),
                       ["rank", "content_id", "avg_position", "ctr", "action_label"]]
print("Weak top-10 picks exposed by the skeptical review:")
display(weak_picks)

# Mechanical leakage proof: score is reproducible from the two declared inputs only.
recomputed = np.where(
    (queue["days_since_last_update"] > 30) & (queue["impressions_90d"] >= 500),
    queue["days_since_last_update"] * np.log1p(queue["impressions_90d"]),
    0.0,
)
assert np.allclose(queue["baseline_score"], recomputed)
assert FORBIDDEN_INPUTS.isdisjoint(RULE_INPUTS)
assert queue["reason_code"].notna().all()
assert not queue["reason_code"].str.contains(r"\|").any()
print(f"Leakage check passed: score uses only {RULE_INPUTS}.")
print("No future-window, trend_pct, trend_direction, label, product flag, URL, or raw query is an input.")

Weak top-10 picks exposed by the skeptical review:


,rank,content_id,avg_position,ctr,action_label
6,7,content_5feee3994adb,39.0,0.01,review_for_refresh
8,9,content_b16bd7307b39,31.0,0.00,review_for_refresh


Leakage check passed: score uses only ['days_since_last_update', 'impressions_90d'].
No future-window, trend_pct, trend_direction, label, product flag, URL, or raw query is an input.


## 5. Self-check

- [x] Two signal checks have visible bucket tables and `n`; staleness is linked to a real FlyRank refresh flag.
- [x] Each signal has a one-word verdict: **CONFIRMED** and **CONFIRMED**.
- [x] One transparent rule produces a score, exactly one reason code per row, and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] All top ten rows have an action, why-it-ranked line, and what-would-make-it-wrong line.
- [x] Weak picks are named and Precision@K is compared with the base rate.
- [x] No future-window or label-derived input enters the rule.
- [x] The notebook runs top to bottom without errors and exposes no client names, URLs, private queries, or secrets.